In [ ]:
# ============================================================
# SETUP: Copia o CSV bruto do Drive e gera o arquivo limpo
# Precisamos fazer isso uma vez por sessão
# ============================================================

import pandas as pd
import os

# Caminho do arquivo bruto no Drive
# Ajuste "Meu Drive" se o arquivo estiver em outra pasta
DRIVE_PATH = "/drive/MyDrive/INFLUD24.csv"

# Verifica se o arquivo existe no Drive
print(f"📁 Arquivo no Drive: {os.path.exists(DRIVE_PATH)}")

In [2]:
# ============================================================
# GOOGLE DRIVE: Monta o Drive para persistir arquivos
# Resolve o problema de perda de arquivos entre sessões
# ============================================================

from google.colab import drive

# Monta o Google Drive em /drive
# Vai pedir autorização — clica em "Conectar ao Google Drive"
drive.mount("/drive")

print("✅ Google Drive montado com sucesso!")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ARQUITETURA: Camada de análise SQL (3/5)
# DuckDB permite rodar SQL diretamente sobre DataFrames
# pandas - sem precisar de um banco de dados externo

# Instalando o DuckDB dentro do Google Colab
!pip install duckdb -q

import duckdb
import pandas as pd

print("✅ DuckDB instalado e importado com sucesso !")
print(f"📦 Versão DuckDB: {duckdb.__version__}")


In [ ]:
# ============================================================
# ENTRADA: Lê o arquivo bruto do Drive, aplica limpeza
# e registra como tabela no DuckDB
# Solução definitiva para não depender de sessão anterior
# ============================================================

import pandas as pd
import duckdb

# Lê o arquivo bruto direto do Google Drive
df_raw = pd.read_csv(
    "/drive/MyDrive/INFLUD24.csv",
    sep=";",             # separador padrão DATASUS
    encoding="latin-1", # encoding padrão arquivos do governo
    low_memory=False
)

# Colunas relevantes para análise — mesmas da etapa de limpeza
COLUNAS_RELEVANTES = [
    "DT_NOTIFIC", "SG_UF_NOT", "ID_MUNICIP",
    "NU_IDADE_N", "CS_SEXO", "CS_RACA", "CS_ESCOL_N",
    "CLASSI_FIN", "EVOLUCAO", "UTI", "SUPORT_VEN",
    "FEBRE", "TOSSE", "DISPNEIA", "SATURACAO",
    "VACINA_COV",
]

# Filtra apenas colunas existentes no dataset
colunas_existentes = [c for c in COLUNAS_RELEVANTES if c in df_raw.columns]

# Aplica a limpeza
df = df_raw[colunas_existentes].copy()

# Cria conexão DuckDB e registra o DataFrame como tabela virtual
con = duckdb.connect()
con.register("srag", df)

print(f"✅ Tabela 'srag' registrada no DuckDB")
print(f"📊 {df.shape[0]:,} linhas | {df.shape[1]} colunas")

In [ ]:
# ============================================================
# ANÁLISE 1: Distribuição de casos por estado
# Responde: quais estados tiveram mais notificações de SRAG?
# ============================================================

query_estados = """
    SELECT
        SG_UF_NOT                    AS estado,
        COUNT(*)                     AS total_casos,
        ROUND(COUNT(*) * 100.0
              / SUM(COUNT(*)) OVER(), 2) AS percentual
    FROM srag
    WHERE SG_UF_NOT IS NOT NULL
    GROUP BY estado
    ORDER BY total_casos DESC
    LIMIT 10
"""

df_estados = con.execute(query_estados).df()
print("=== Top 10 estados com mais casos de SRAG ===")
print(df_estados)

In [ ]:
# ============================================================
# ANÁLISE 2: Evolução clínica dos pacientes
# Versão corrigida — trata coluna EVOLUCAO como texto
# pois o DATASUS às vezes armazena números como string
# ============================================================

query_evolucao = """
    SELECT
        CASE CAST(EVOLUCAO AS VARCHAR)
            WHEN '1' THEN 'Cura'
            WHEN '2' THEN 'Óbito por SRAG'
            WHEN '3' THEN 'Óbito por outras causas'
            ELSE 'Não informado'
        END                          AS evolucao,
        COUNT(*)                     AS total,
        ROUND(COUNT(*) * 100.0
              / SUM(COUNT(*)) OVER(), 2) AS percentual
    FROM srag
    GROUP BY EVOLUCAO
    ORDER BY total DESC
"""

df_evolucao = con.execute(query_evolucao).df()
print("=== Evolução clínica dos pacientes ===")
print(df_evolucao)

In [ ]:
# ============================================================
# ANÁLISE 3: Distribuição por faixa etária
# Versão corrigida — converte NU_IDADE_N explicitamente
# para número antes de comparar
# ============================================================

query_idade = """
    SELECT
        CASE
            WHEN TRY_CAST(NU_IDADE_N AS INTEGER) < 5   THEN '0-4 anos'
            WHEN TRY_CAST(NU_IDADE_N AS INTEGER) < 12  THEN '5-11 anos'
            WHEN TRY_CAST(NU_IDADE_N AS INTEGER) < 18  THEN '12-17 anos'
            WHEN TRY_CAST(NU_IDADE_N AS INTEGER) < 40  THEN '18-39 anos'
            WHEN TRY_CAST(NU_IDADE_N AS INTEGER) < 60  THEN '40-59 anos'
            WHEN TRY_CAST(NU_IDADE_N AS INTEGER) < 80  THEN '60-79 anos'
            ELSE '80+ anos'
        END                       AS faixa_etaria,
        COUNT(*)                  AS total_casos
    FROM srag
    WHERE NU_IDADE_N IS NOT NULL
    GROUP BY faixa_etaria
    ORDER BY total_casos DESC
"""

df_idade = con.execute(query_idade).df()
print("=== Casos por faixa etária ===")
print(df_idade)